# La Liga Match Predictor
## Project Overview
This project builds a predictive machine learning pipeline to determine the ourcomes of La Liga football matches. It uses historical data from matches (including performance metrics like shots, shots on target, and possession), the goal is to successfully predict whether a given team will win their upcoming match.

Rather than just settling for a baseline model, in this project we will take an iterative engineering approach, we will have the following project workflow: 

1. **Data Cleaning & Exploration**
2. **Establishing the Baseline Model**
3. **Feature Engineering via Rolling Averages**
4. **Aligning Two-Sided Match Predictions**

### Reading Match Data into Pandas Dataframe

In [1]:
import pandas as pd 

# Reading in file
matches = pd.read_csv("matches.csv", index_col=0)

# Sanity Check
matches.head()

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,opp formation,referee,match report,notes,sh,sot,pk,pkatt,season,team
0,2025-08-16,19:30 (10:30),La Liga,Matchweek 1,Sat,Away,W,3,0,Mallorca,...,4-2-3-1,José Luis Munuera,Match Report,NaN,24,8,0,0,2026,Barcelona
1,2025-08-23,21:30 (12:30),La Liga,Matchweek 2,Sat,Away,W,3,2,Levante,...,5-4-1,Alejandro Hernández,Match Report,NaN,26,10,0,0,2026,Barcelona
2,2025-08-31,21:30 (12:30),La Liga,Matchweek 3,Sun,Away,D,1,1,Rayo Vallecano,...,4-4-2,Mateo Busquets,Match Report,NaN,12,3,1,1,2026,Barcelona
3,2025-09-14,21:00 (12:00),La Liga,Matchweek 4,Sun,Home,W,6,0,Valencia,...,5-3-2,Guillermo Cuadra,Match Report,NaN,24,10,0,0,2026,Barcelona
5,2025-09-21,21:00 (12:00),La Liga,Matchweek 5,Sun,Home,W,3,0,Getafe,...,5-3-2,Ricardo de Burgos,Match Report,NaN,16,7,0,0,2026,Barcelona


### Cleaning Our Data for Machine Learning
Currently if we look at our datatypes, many of the columns we have are being stored as "objects" (which usually means text). So in order to be able to use this data for our machine learning model, we need to clean our data, and turn these text columns into numbers. 

In [2]:
# Converting our date from object -> datetime
matches['date'] = pd.to_datetime(matches['date'])

# Sanity Check
matches.dtypes

date             datetime64[ns]
time                     object
comp                     object
round                    object
day                      object
venue                    object
result                   object
gf                        int64
ga                        int64
opponent                 object
poss                      int64
attendance                int64
captain                  object
formation                object
opp formation            object
referee                  object
match report             object
notes                   float64
sh                        int64
sot                       int64
pk                        int64
pkatt                     int64
season                    int64
team                     object
dtype: object

### Creating Predictors for Machine Learning
Now that our data is a bit cleaner, we need to create our "predictors." Predictors are like clues for our machine learning model will use to guess who wins a match.

Machine learning models only understand numbers so we still need to convert take some of our text columns (like who the opponent is, or if the team is playing at home) and turn them into number codes. We create the following columns:

1. **Home or Away (Venue Code)**:
<br>Playing at home gives teams a big advantage. We need to turn the `venue` column (which says "Home" or "Away") into numbers. We will assign a 0 for away games and a 1 for home games.

In [3]:
matches["venue_code"] = matches["venue"].astype("category").cat.codes

2. **Opponent Code**:
<br>Just like the venue, the model needs to know who the team is playing against. We will give every opponent a unique ID number. For example, Mallorca might become 12, and Levante might become 11.

In [4]:
matches["opp_code"] = matches["opponent"].astype("category").cat.codes

3. **Match Hour**:
<br>Some teams might play better at certain times of the day. Our time column looks like this: 16:30 (4:30 PM). We want to drop the colon and the minutes so we are just left with the hour (16) as a whole number.

In [5]:
matches["hour"] = matches["time"].str.replace(":.+", "", regex=True).astype("int")

4. **Day of the Week (Day Code)**:
<br>Another factor on how well a team plays, might be the day of the week. We can use the datetime column we fixed earlier to figure out what day of the week the match was on. This will give us a number from 0 (Monday) to 6 (Sunday).

In [6]:
matches["day_code"] = matches["date"].dt.day_of_week

5. **Setting Up the Target**:
<br>Lastly, we want to know which team won. Right now, the `result` columns says `W` (Win), `L` (Loss), or `D` (Draw). We will create a new column called `target`. If the team won, it gets a 1. If they lost or tied, it gets a 0.

In [7]:
matches["target"] = (matches["result"] == "W").astype("int")

# Sanity Check
matches.head()

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,sot,pk,pkatt,season,team,venue_code,opp_code,hour,day_code,target
0,2025-08-16,19:30 (10:30),La Liga,Matchweek 1,Sat,Away,W,3,0,Mallorca,...,8,0,0,2026,Barcelona,0,12,19,5,1
1,2025-08-23,21:30 (12:30),La Liga,Matchweek 2,Sat,Away,W,3,2,Levante,...,10,0,0,2026,Barcelona,0,11,21,5,1
2,2025-08-31,21:30 (12:30),La Liga,Matchweek 3,Sun,Away,D,1,1,Rayo Vallecano,...,3,1,1,2026,Barcelona,0,15,21,6,0
3,2025-09-14,21:00 (12:00),La Liga,Matchweek 4,Sun,Home,W,6,0,Valencia,...,10,0,0,2026,Barcelona,1,20,21,6,1
5,2025-09-21,21:00 (12:00),La Liga,Matchweek 5,Sun,Home,W,3,0,Getafe,...,7,0,0,2026,Barcelona,1,7,21,6,1


### Building and Evaluating Our First Machine Learning Model
Now we can start building our first model, we will use a **Random Forest Classifier**. This model is made up of many small "decision trees" that work together.

It is great for football data because it doesn't assume straight-line patterns. For example, an opponent code of 12 (Mallorca) isn't "higher" or "better" than 11 (Levante); they are just different. A Random Forest understands this, whereas other simple models might get confused. Let's start by:

1. **Setting Up the Model**:<br>
We import the Random Forest model and set a few settings:
- `n_estimators`: <br>The number of individual decision trees we want to build. More trees can be more accurate but take longer to run.
- `min_samples_split`: <br>Keeps the model from memorizing the training data too closely (overfitting).
- `random_state`: <br>Ensures we get the exact same results every time we run the code.

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)

2. **Split the Data (Train vs. Test)**:
<br>Because football matches happen over time, we need to split up our data. We cannot train our model on future games to predict past games.
<br>We will train our model on matches played before 2026, and test how well it predicts matches played in 2026. This is like a "closed-book" test, we evaluate the model on data it has never seen before.

In [9]:
# Train on older matches (before 2026)
train = matches[matches["date"] < '2026-01-01']

# Test on newer matches (2026 onward)
test = matches[matches["date"] > '2026-01-01']

# Choose our numeric clues
predictors = ["venue_code", "opp_code", "hour", "day_code"]

3. **Train the Model and Predict**:
<br>We "fit" (train) the model using our training clues and targets, then ask it to make predictions on our test set.

In [10]:
# Training the model
rf.fit(train[predictors], train["target"])

# Make predictions on the test set
preds = rf.predict(test[predictors])

4. **Check Overall Accuracy**:
<br>First, we check general accuracy. What percentage of our total predictions (wins, losses, and draws) did we get right?

In [11]:
from sklearn.metrics import accuracy_score

# Passing in actual scores and predicted scores
acc = accuracy_score(test["target"], preds)

print(f"Overall Accuracy: {acc:.1%}")

Overall Accuracy: 62.4%


We got an accuracy score of **~62%**, but let's take a closer look and see in what situations our accuracy was high vs. low.

5. **Taking a Closer Look**:
<br>We need to see what happened when we specifically guessed that a team would win. We will create a "cross-tab" (a table comparing our predictions against real outcomes).

In [12]:
from sklearn.metrics import accuracy_score

# Passing in actual scores and predicted scores
acc = accuracy_score(test["target"], preds)

print(f"Overall Accuracy: {acc:.1%}")

Overall Accuracy: 62.4%


**What we noticed**: 
<br>When we predicted a loss or draw (0), we were correct 212 times and wrong 110 times. But when we predicted a win (1), we were wrong 47 times and right 49 times.

6. **Measure Precision**:
<br>We want to create this model to predict winners, so we should only care about Precision. We should ask ourselves: **When we predict a team will win, how often do they actually win?**<br>In order to answer this we will use the `precision_score` metric. 

In [13]:
from sklearn.metrics import precision_score

# Calculating precision
precision = precision_score(test["target"], preds)
print(f"Win Prediction Precision: {precision:.1%}")

Win Prediction Precision: 51.0%


**Result**: <br>Our precision is only **51%**.

**Next Steps**
<br>A **51%** win precision is not great. In the next phase, we will improve this by giving our model better predictors, like rolling averages of how well teams played in their last few games.

### Improving Precision with Rolling Averages
Right now, our model only knows basic facts like where the match is played and who the opponent is. It does not know if a team is on a winning streak or if they have been playing terribly.

To fix this, we will get the rolling averages. This means for any upcoming match, we will look at how that team performed in their previous 3 matches (how many goals they scored, how many shots they took, etc.) and use that history as a clue for the model.

1. **Create the Rolling Averages Function**:
<br>In order to calulate the Rolling Averages accurately we must do the following: 
- **Group by Team**:<br>We must separate the data by team so we do not mix stats between teams.
- **Look Backward, Not Forward**:<br>When predicting a match in Week 4, we only want the average of Weeks 1, 2, and 3. We must tell pandas to leave out the current week's data so the model doesn't secretly peek into the future.
- **Drop Missing Data**:<br>For the first few games of the season (Weeks 1, 2, and 3), there aren't 3 past games to look back on. Pandas will mark these as empty values (NaN). Because our machine learning model cannot handle empty values, we will remove those early rows.

In [14]:
# Grouping by Team
grouped_matches = matches.groupby("team")

# Choosing Specific Team
group = grouped_matches.get_group("Real Madrid").sort_values("date")

def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    # 'Take the current week out and ignore it when calculating these averages'
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

2. **Choosing what Stats we should Track**
<br>We will select specific metrics that show how good a teams attack is doing and how good they are defensively.

In [15]:
cols = ['gf', 'ga', 'sot']
new_cols = [f"{c}_rolling" for c in cols]

rolling_averages(group, cols, new_cols)

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,season,team,venue_code,opp_code,hour,day_code,target,gf_rolling,ga_rolling,sot_rolling
4,2024-09-01,21:30 (12:30),La Liga,Matchweek 4,Sun,Home,W,2,0,Real Betis,...,2025,Real Madrid,1,16,21,6,1,1.666667,0.666667,7.333333
5,2024-09-14,21:00 (12:00),La Liga,Matchweek 5,Sat,Away,W,2,0,Real Sociedad,...,2025,Real Madrid,0,18,21,5,1,2.000000,0.333333,8.000000
7,2024-09-21,21:00 (12:00),La Liga,Matchweek 6,Sat,Home,W,4,1,Espanyol,...,2025,Real Madrid,1,6,21,5,1,1.666667,0.333333,7.000000
8,2024-09-24,21:00 (12:00),La Liga,Matchweek 7,Tue,Home,W,3,2,Alavés,...,2025,Real Madrid,1,0,21,1,1,2.666667,0.333333,9.000000
9,2024-09-29,21:00 (12:00),La Liga,Matchweek 8,Sun,Away,D,1,1,Atlético Madrid,...,2025,Real Madrid,0,2,21,6,0,3.000000,1.000000,8.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52,2026-05-03,21:00 (12:00),La Liga,Matchweek 34,Sun,Away,W,2,0,Espanyol,...,2026,Real Madrid,0,6,21,6,1,1.333333,1.000000,8.333333
53,2026-05-10,21:00 (12:00),La Liga,Matchweek 35,Sun,Away,L,0,2,Barcelona,...,2026,Real Madrid,0,3,21,6,0,1.666667,0.666667,7.000000
54,2026-05-14,21:30 (12:30),La Liga,Matchweek 36,Thu,Home,W,2,0,Oviedo,...,2026,Real Madrid,1,14,21,3,1,1.000000,1.000000,4.666667
55,2026-05-17,19:00 (10:00),La Liga,Matchweek 37,Sun,Away,W,1,0,Sevilla,...,2026,Real Madrid,0,19,19,6,1,1.333333,0.666667,4.333333


4. **Apply to All Teams and Clean the Index**:
<br>Now we apply our function to every team in the dataset. After grouping and applying functions, pandas adds extra layers to our dataframe index (labels), so we will clean those up to keep our dataframe simple and easy to read.

In [16]:
# We grouped our original matches df by team, and we applied the rolling averages function to each team df we created
matches_rolling = matches.groupby("team").apply(
    lambda x: rolling_averages(x, cols, new_cols), 
    include_groups=False
)

matches_rolling = matches_rolling.reset_index(level="team")

# Make sure every row has a unique index number from 0 onwards
matches_rolling = matches_rolling.reset_index(drop=True)

### Retraining Our Machine Learning Model
Now that we have our new rolling averages ready. Instead of writing out the code to train, test, and score our model over and over again, we will wrap everything into a single reusable function. This will make it much faster to experiment with new predictors.

1. **Create a Reusable Prediction Function**:
<br>This function takes our data and a list of predictors, splits the data by time, trains the Random Forest model, makes predictions, and calculates our precision score.

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

def make_predictions(data, predictors):
    # Instantiate a fresh model inside the function to avoid global overwrites
    model = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)
    
    # Fixed the edge-case by adding >= 
    train = data[data["date"] < '2026-01-01']
    test = data[data["date"] >= '2026-01-01']
    
    # Fitting the fresh model
    model.fit(train[predictors], train["target"])
    preds = model.predict(test[predictors])
    
    # Combining predictions and actuals together 
    combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
    
    # Calculating precision
    precision = precision_score(test["target"], preds)
    
    return combined, precision

2. **Run the Model**:
<br>Now we run our function using our original basic predictors plus the new 3-match rolling average stats we just made.

In [18]:
# Execute the function using our dataframe with rolling averages
combined, precision = make_predictions(matches_rolling, predictors + new_cols)
baseline_combined, baseline_precision = make_predictions(matches_rolling, predictors)

print(f"True Baseline Precision: {baseline_precision:.1%}")
print(f"New Precision Score: {precision:.1%}")

True Baseline Precision: 50.5%
New Precision Score: 50.0%


3. Add Match Details Back to Our Predictions
Right now, our `combined` DataFrame only shows 0s and 1s. It is hard to see which specific teams the model is getting right or wrong.

To fix this, we will merge key information (like dates, team names, opponents, and actual results) back into our predictions using the row index numbers.

In [19]:
# Merge team details into our predictions based on the matching index rows
combined = combined.merge(matches_rolling[["date", "team", "opponent", "result"]], left_index=True, right_index=True)

# Take a look at the updated predictions table
combined.head(10)

,actual,predicted,date,team,opponent,result
52,0,0,2026-01-04,Alaves,Oviedo,D
53,0,0,2026-01-10,Alaves,Villarreal,L
54,0,0,2026-01-18,Alaves,Atlético Madrid,L
55,1,0,2026-01-25,Alaves,Real Betis,W
56,1,0,2026-01-30,Alaves,Espanyol,W
57,0,0,2026-02-08,Alaves,Getafe,L
58,0,1,2026-02-14,Alaves,Sevilla,D
59,0,0,2026-02-23,Alaves,Girona,D
60,0,0,2026-02-27,Alaves,Levante,L
61,0,0,2026-03-08,Alaves,Valencia,L


#### Why Did Our Score Go Down?

**Result**: <br>Our true baseline precision dropped to **50.5%**, and our new rolling averages dropped it further to **50.0%**.

We expected that adding rolling averages would instantly make our model smarter. So, what happened? In machine learning, more data doesn't always mean *better* data. Here are the two reasons our score went backward:

* **1. We Took a Different Test (Missing Data)** <br>To calculate a 3-game rolling average, we had to delete the first 3 matches of the season for every single team (since they had no previous games to average yet). Because we deleted roughly 60 matches from our dataset, our model was evaluated on a slightly different, harder set of games than our original 51% run.
* **2. Too Much Noise (Overfitting)** <br>We initially fed the model stats like Penalty Kicks. Penalties are rare and highly random. Our model tried to find patterns in that randomness, got confused, and made worse predictions. This is a classic machine learning trap called "overfitting."

### Combining Home and Away Predictions
Every football match has two sides: a home team and an away team. Right now, our dataset treats them completely separately, giving each team its own row. Because of this, our model guesses the outcome for each team independently. This can lead to impossible situations; like the model predicting that both teams will win the exact same match.

To fix this, we need to join the home rows and away rows together so we can look at both sides of a match at the same time.

1. **Fixing Mismatched Team Names**:
<br>Before we can link the match rows, we have a small problem. The team names are written differently depending on which column they are in.

We will create a smart dictionary to automatically fix these mismatched names. If a name is on our mismatch list, it gets fixed.

In [20]:
# 1. Create our dictionary class that ignores missing keys instead of crashing
class MissingDict(dict):
    __missing__ = lambda self, key: key

# 2. La Liga mappings (URL names -> FBref Match Log names with accents)
# Note: Depending on the exact years you scraped, you might need to add to this list!
map_values = {
    "Atletico Madrid": "Atlético Madrid",
    "Cadiz": "Cádiz",
    "Alaves": "Alavés",
    "Almeria": "Almería",
    "Leganes": "Leganés",
    "Deportivo La Coruna": "Deportivo La Coruña",
    "Malaga": "Málaga"
} 

mapping = MissingDict(**map_values)

# 3. Create the translated column
combined["new_team"] = combined["team"].map(mapping)

2. **Lining Up Both Sides of the Match**:
Now that the team names match perfectly, we can merge our predictions table with itself. This lines up the data so that for any given date, we can see Team A's prediction right next to Team B's prediction.

In [21]:
# 3. Create the translated column
combined["new_team"] = combined["team"].map(mapping)

# 4. Merge the dataset on itself to align Team A's prediction with Team B's prediction
merged = combined.merge(
    combined, 
    left_on=["date", "new_team"], 
    right_on=["date", "opponent"]
)

3. **Filtering for High-Confidence Predictions**:
<br>Now we can look for the matches where our model is extra confident. We will filter our data to only look at games where the model predicted Team A would win (1) and predicted Team B would lose (0).

In [22]:
# 5. Isolate our High-Confidence bets (Predicted A to win, Predicted B to lose)
high_confidence = merged[(merged["predicted_x"] == 1) & (merged["predicted_y"] == 0)]

# 6. See how many we got right (1) vs wrong (0)
print("Actual Results of High-Confidence Bets:")
print(high_confidence["actual_x"].value_counts())

Actual Results of High-Confidence Bets:
actual_x
1    45
0    41
Name: count, dtype: int64


**Conclusion**
<br>By looking at both sides of the match, we filter out the games where the model gave conflicting or unsure guesses. When we look strictly at these high-confidence matchups, our win prediction precision rises to 52.3%.

We managed to significantly improve our model's accuracy without changing any machine learning mathematical settings, we just used a bit of logic to clean and combine our data.

In [23]:
from sklearn.metrics import precision_score
final_precision = precision_score(high_confidence["actual_x"], high_confidence["predicted_x"])

print(f"\nFinal Dual-Sided Precision Score: {final_precision:.1%}")


Final Dual-Sided Precision Score: 52.3%
